<a href="https://colab.research.google.com/github/abdullahamjad20/Artifical-Intelligence-Lab/blob/main/AI%20Lab%201/task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import random

MOVE_PENALTY = 1
STEPS = 20


def random_initial_state(rng):
    return rng.choice("AB"), rng.random() < 0.5, rng.random() < 0.5


def step(state, action):
    loc, dirt_a, dirt_b = state
    dirt = {"A": dirt_a, "B": dirt_b}
    penalty = 0
    if action == "Suck":
        dirt[loc] = False
    elif action == "Left":
        loc, penalty = "A", MOVE_PENALTY
    elif action == "Right":
        loc, penalty = "B", MOVE_PENALTY
    reward = (not dirt["A"]) + (not dirt["B"]) - penalty
    return (loc, dirt["A"], dirt["B"]), reward


class SimpleReflexAgent:
    def act(self, percept):
        loc, dirty = percept
        if dirty:
            return "Suck"
        return "Right" if loc == "A" else "Left"


class RandomAgent:
    def __init__(self, rng):
        self.rng = rng

    def act(self, percept):
        return self.rng.choice(["Left", "Right", "Suck"])


def run_episode(agent, state, steps=STEPS):
    total = 0
    for _ in range(steps):
        loc, dirt_a, dirt_b = state
        percept = (loc, dirt_a if loc == "A" else dirt_b)
        state, reward = step(state, agent.act(percept))
        total += reward
    return total


seed, n_states = 42, 10
rng = random.Random(seed)
states = [random_initial_state(rng) for _ in range(n_states)]

print(f"{'Run':<4}{'Initial state':<40}{'Reflex':<8}{'Random':<8}")
reflex_scores, random_scores = [], []
for i, s in enumerate(states, 1):
    r = run_episode(SimpleReflexAgent(), s)
    q = run_episode(RandomAgent(random.Random(seed + i)), s)
    reflex_scores.append(r)
    random_scores.append(q)
    desc = f"loc={s[0]}, A_dirty={s[1]}, B_dirty={s[2]}"
    print(f"{i:<4}{desc:<40}{r:<8}{q:<8}")

print(f"\nAverage Simple Reflex score : {sum(reflex_scores)/n_states:.2f}")
print(f"Average Random agent score  : {sum(random_scores)/n_states:.2f}")

# Extra check: 10,000 runs, with and without the move penalty
for pen in (1, 0):
    MOVE_PENALTY = pen
    big = random.Random(999)
    br = bm = 0
    N = 10000
    for _ in range(N):
        s = random_initial_state(big)
        br += run_episode(SimpleReflexAgent(), s)
        bm += run_episode(RandomAgent(big), s)
    print(f"[{N} runs, move penalty={pen}] reflex={br/N:.2f}, random={bm/N:.2f}")

print("\nWith no move penalty the reflex agent clearly wins. With a move penalty "
      "it keeps paying\n-1 every step after cleaning, so the random agent can "
      "score about the same or higher.")

Run Initial state                           Reflex  Random  
1   loc=A, A_dirty=True, B_dirty=True       20      13      
2   loc=A, A_dirty=True, B_dirty=True       20      14      
3   loc=A, A_dirty=False, B_dirty=True      20      8       
4   loc=A, A_dirty=True, B_dirty=False      21      25      
5   loc=A, A_dirty=False, B_dirty=False     20      25      
6   loc=B, A_dirty=True, B_dirty=False      20      26      
7   loc=A, A_dirty=False, B_dirty=True      20      23      
8   loc=B, A_dirty=True, B_dirty=True       20      17      
9   loc=B, A_dirty=True, B_dirty=True       20      21      
10  loc=B, A_dirty=False, B_dirty=False     20      24      

Average Simple Reflex score : 20.10
Average Random agent score  : 19.60
[10000 runs, move penalty=1] reflex=20.25, random=20.62
[10000 runs, move penalty=0] reflex=39.26, random=33.95

With no move penalty the reflex agent clearly wins. With a move penalty it keeps paying
-1 every step after cleaning, so the random agent can s